In [17]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

fuel_df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("Files/raw/batch/form41/fuel_cost_consumption_2024_jan.csv")
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 19, Finished, Available, Finished, False)

In [18]:
print("Rows :", fuel_df.count())

fuel_df.printSchema()

display(fuel_df.limit(10))

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 20, Finished, Available, Finished, False)

Rows : 53
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- AIRLINE_ID: integer (nullable = true)
 |-- UNIQUE_CARRIER: string (nullable = true)
 |-- CARRIER_NAME: string (nullable = true)
 |-- TDOMT_GALLONS: double (nullable = true)
 |-- TINT_GALLONS: double (nullable = true)
 |-- TOTAL_GALLONS: double (nullable = true)
 |-- TDOMT_COST: double (nullable = true)
 |-- TINT_COST: double (nullable = true)
 |-- TOTAL_COST: double (nullable = true)



SynapseWidget(Synapse.DataFrame, a14db0bf-21ac-4509-8fe8-3d45f175679e)

In [19]:
import re

def standardize_columns(df):
    for col_name in df.columns:
        new_name = (
            col_name.strip()
                    .lower()
                    .replace(" ", "_")
        )
        new_name = re.sub(r'[^a-z0-9_]', '', new_name)
        df = df.withColumnRenamed(col_name, new_name)
    return df

fuel_df = standardize_columns(fuel_df)

fuel_df.printSchema()

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 21, Finished, Available, Finished, False)

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- airline_id: integer (nullable = true)
 |-- unique_carrier: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- tdomt_gallons: double (nullable = true)
 |-- tint_gallons: double (nullable = true)
 |-- total_gallons: double (nullable = true)
 |-- tdomt_cost: double (nullable = true)
 |-- tint_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)



In [20]:
dim_airline = spark.read.table("dim_airline")

dim_date = spark.read.table("dim_date")

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 22, Finished, Available, Finished, False)

In [21]:
dim_airline.printSchema()

dim_date.printSchema()

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 23, Finished, Available, Finished, False)

root
 |-- airline_key: integer (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- airline_name: string (nullable = true)
 |-- country: string (nullable = true)

root
 |-- flight_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- date_key: integer (nullable = true)



In [22]:
fuel_df = fuel_df.withColumn(
    "operation_date",
    to_date(
        concat_ws("-", col("year"), col("month"), lit(1))
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 24, Finished, Available, Finished, False)

In [23]:
airline_lookup = (
    dim_airline
    .select(
        "airline_key",
        col("iata_code").alias("unique_carrier")
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 25, Finished, Available, Finished, False)

In [24]:
fuel_df = (
    fuel_df
    .join(
        airline_lookup,
        on="unique_carrier",
        how="left"
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 26, Finished, Available, Finished, False)

In [25]:
print(
    "Null Airline Keys:",
    fuel_df.filter(col("airline_key").isNull()).count()
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 27, Finished, Available, Finished, False)

Null Airline Keys: 13


In [26]:
date_lookup = (
    dim_date
    .select(
        "date_key",
        col("flight_date").alias("operation_date")
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 28, Finished, Available, Finished, False)

In [27]:
fuel_df = (
    fuel_df
    .join(
        date_lookup,
        on="operation_date",
        how="left"
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 29, Finished, Available, Finished, False)

In [28]:
print(
    "Null Date Keys:",
    fuel_df.filter(col("date_key").isNull()).count()
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 30, Finished, Available, Finished, False)

Null Date Keys: 0


In [29]:
display(
    fuel_df.select(
        "unique_carrier",
        "airline_key",
        "operation_date",
        "date_key",
        "total_gallons",
        "total_cost"
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da293275-af8b-48b7-b181-5799d6b83782)

In [30]:
missing_airlines = (
    fuel_df
    .filter(col("airline_key").isNull())
    .select(
        "unique_carrier",
        "carrier_name"
    )
    .distinct()
    .orderBy("unique_carrier")
)

display(missing_airlines)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dbd7bfa3-84b1-470d-9b45-9f86dd04a9de)

In [31]:
display(
    dim_airline
    .select("iata_code", "airline_name")
    .orderBy("iata_code")
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8066cfc-ec66-44c0-8126-a09fae2350d0)

In [32]:
fuel_df = fuel_df.filter(col("airline_key").isNotNull())

print("Rows After Filtering:", fuel_df.count())

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 34, Finished, Available, Finished, False)

Rows After Filtering: 54


In [33]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy(
    "date_key",
    "airline_key"
)

fact_fuel_consumption = (
    fuel_df
    .withColumn(
        "fuel_key",
        row_number().over(window_spec)
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 35, Finished, Available, Finished, False)

In [34]:
fact_fuel_consumption = (
    fact_fuel_consumption
    .select(
        "fuel_key",
        "date_key",
        "airline_key",

        col("total_gallons"),
        col("tdomt_gallons").alias("domestic_gallons"),
        col("tint_gallons").alias("international_gallons"),

        col("total_cost"),
        col("tdomt_cost").alias("domestic_cost"),
        col("tint_cost").alias("international_cost")
    )
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 36, Finished, Available, Finished, False)

In [35]:
print("Rows:", fact_fuel_consumption.count())

fact_fuel_consumption.printSchema()

display(fact_fuel_consumption)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 37, Finished, Available, Finished, False)

Rows: 54
root
 |-- fuel_key: integer (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- airline_key: integer (nullable = true)
 |-- total_gallons: double (nullable = true)
 |-- domestic_gallons: double (nullable = true)
 |-- international_gallons: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- domestic_cost: double (nullable = true)
 |-- international_cost: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 52486f7e-f05d-4b28-8c80-6d9dd397e0a7)

In [36]:
fact_fuel_consumption.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_fuel_consumption")

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 38, Finished, Available, Finished, False)

In [37]:
spark.read.table("fact_fuel_consumption").count()

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 39, Finished, Available, Finished, False)

54

In [38]:
print(
    "Null Airline Keys:",
    fact_fuel_consumption.filter(col("airline_key").isNull()).count()
)

StatementMeta(, 31212e75-def6-4a95-ad1b-2d030ceb66f2, 40, Finished, Available, Finished, False)

Null Airline Keys: 0
